# sensa-ai: Colab'da değerlendirme ve eğitim (yalnız sayı)

**Güvenlik (D-001):** Bu defter hiçbir görüntü göstermez. Soldaki **Dosyalar** panelinde `data/pozitif` klasörünü açma, görüntülere tıklama (Colab önizleme açar). Çıktılar yalnız tablo ve sayı.

Başlamadan: Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**. Sol menüde 🔑 **Gizli anahtarlar** → `ROBOFLOW_API_KEY` ekle ve bu deftere erişim ver.

In [ ]:
!git clone -q -b claude/repo-yapisi-ve-plan https://github.com/sulku-AI/sensa-ai.git
%cd sensa-ai
!pip install -q -e ".[models,data,train]" sentencepiece protobuf
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import userdata
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
!python -m tools.fetch_datasets blood_seg_joost blood_pure_blur
!python -m tools.fetch_negatives --per-class 40

## 1. Tanı: kapı mı dedektör mü kaçırıyor?

In [ ]:
POS = '--pos data/pozitif/blood_seg_joost --pos data/pozitif/blood_pure_blur'
!python -m tools.evaluate $POS --no-gate --device cuda
!python -m tools.evaluate $POS --no-gate --box-threshold 0.1 --device cuda
!python -m tools.evaluate $POS --gate-threshold 0.15 --box-threshold 0.1 --device cuda

## 2. Denetimli eğitim: RF-DETR-Seg Nano (Apache 2.0)

T4'te birkaç saat sürebilir. Colab ücretsiz sürümü ~12 saatte ya da hareketsizlikte oturumu kapatabilir.

In [ ]:
!python -m tools.train_rfdetr prepare --src data/pozitif/blood_seg_joost data/pozitif/blood_pure_blur --neg data/negatif/coco
!python -m tools.train_rfdetr train --epochs 30 --batch 8 --accum 2 --device cuda

## 3. Sonuçları kaydet

Yalnız sayı dosyaları (`experiments/results/*.md`) ve ağırlıklar. Ağırlık dosyası görüntü değildir.

In [ ]:
!cat experiments/results/*_eval_*.md
!ls -la models/ 2>/dev/null